In [16]:
!pip install --force-reinstall --no-cache-dir opentelemetry-api==1.45.2 opentelemetry-sdk==1.45.2

  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.16.0
    Uninstalling typing_extensions-4.16.0:
      Successfully uninstalled typing_extensions-4.16.0
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.45.2
    Uninstalling opentelemetry-api-1.45.2:
      Successfully uninstalled opentelemetry-api-1.45.2
  Attempting uninstall: opentelemetry-semantic-conventions
    Found existing installation: opentelemetry-semantic-conventions 0.66b2
    Uninstalling opentelemetry-semantic-conventions-0.66b2:
      Successfully uninstalled opentelemetry-semantic-conventions-0.66b2
  Attempting uninstall: opentelemetry-sdk
    Found existing installation: opentelemetry-sdk 1.45.2
    Uninstalling opentelemetry-sdk-1.45.2:
      Successfully uninstalled opentelemetry-sdk-1.45.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [opentelemetry-sdk]
ERROR: pip's dependency resolver does not currently take into accoun

In [1]:
!pip install -q langchain langgraph langchain-groq chromadb

In [2]:
import chromadb
import langchain
import langgraph
import langchain_groq

from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

print("All imports successful!")

All imports successful!


In [5]:
import os
from google.colab import userdata
from langchain_groq import ChatGroq

print("Imports successful!")

Imports successful!


In [6]:
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.3
)

print("Groq LLM connected successfully!")

Groq LLM connected successfully!


In [7]:
response = llm.invoke("Hello! Introduce yourself.")

print(response.content)

Hello! I’m ChatGPT, a large language model created by OpenAI. I’m here to help answer questions, brainstorm ideas, explain concepts, or just chat about almost anything you’re curious about. Whether you need help with writing, coding, learning a new topic, or just want a friendly conversation, feel free to ask!


In [8]:
from langchain_core.tools import tool

@tool
def get_price(item: str) -> str:
    """Get the price of bakery products."""
    prices = {
        "plum cake": 500,
        "chocolate cake": 650,
        "cupcake": 80
    }

    item = item.lower()

    for name, price in prices.items():
        if name in item:
            return f"{name} costs Rs {price}."

    return "Price not available. I'll check with the owner."


@tool
def is_open(day: str) -> str:
    """Check bakery opening hours."""
    hours = {
        "monday": "9 am to 8 pm",
        "tuesday": "9 am to 8 pm",
        "wednesday": "9 am to 8 pm",
        "thursday": "9 am to 8 pm",
        "friday": "9 am to 8 pm",
        "saturday": "9 am to 8 pm",
        "sunday": "10 am to 6 pm"
    }

    return hours.get(day.lower(), "I'll check with the owner.")


@tool
def log_order(name: str, item: str, quantity: int, pickup_date: str) -> str:
    """Save a customer bakery order."""
    return (
        f"Order recorded for {name}: "
        f"{quantity} {item}, pickup date {pickup_date}."
    )

print("Bakery tools created successfully!")

Bakery tools created successfully!


In [9]:
client = chromadb.Client()

faq = client.get_or_create_collection(
    name="sweet_crumbs_faq"
)

FAQS = {
    "delivery": "We deliver within 5 km of the bakery. Delivery is Rs 50, free for orders above Rs 1000.",
    "payment": "We accept UPI and cash. Orders above Rs 1000 need a 50% advance by UPI.",
    "cancellation": "Cancellation is free up to 12 hours before pickup. After that, the advance is not refunded.",
    "photo_cake": "Photo cakes and custom designs need 48 hours notice.",
    "notice": "Regular orders need 24 hours notice.",
    "eggless": "All our cakes have eggless versions at the same price.",
    "allergens": "Plum cake contains nuts and dried fruit. Cupcakes and chocolate cake are nut-free.",
    "storage": "Keep cakes in the fridge and eat them within 3 days.",
    "message": "A message on the cake is free, up to 20 letters.",
    "candles": "Candles are free with every cake.",
    "discounts": "Bulk-order discounts need the owner's approval.",
    "pickup": "Pickup is Monday to Saturday 9 am to 8 pm, Sunday 10 am to 6 pm.",
    "plum_cake": "Plum cake costs Rs 500.",
    "chocolate_cake": "Chocolate cake costs Rs 650.",
    "cupcakes": "Cupcakes cost Rs 80 each.",
    "orders": "Orders must include customer name, item, quantity and pickup date."
}

faq.upsert(
    ids=list(FAQS.keys()),
    documents=list(FAQS.values())
)

print("Total facts stored:", faq.count())

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 53.2MiB/s]


Total facts stored: 16


In [10]:
results = faq.query(
    query_texts=["Can you deliver to my house?"],
    n_results=2
)

print(results["documents"][0])

['We deliver within 5 km of the bakery. Delivery is Rs 50, free for orders above Rs 1000.', 'Regular orders need 24 hours notice.']


In [11]:
@tool
def search_faq(question: str) -> str:
    """
    Search Sweet Crumbs policies and FAQs.

    Topics:
    delivery, payment, cancellation, photo cakes,
    advance notice, eggless cakes, allergens,
    storage, cake messages, candles, discounts,
    pickup timings, prices and orders.
    """

    results = faq.query(
        query_texts=[question],
        n_results=3
    )

    documents = results["documents"][0]

    if not documents:
        return "No relevant information found."

    return "\n".join(documents)

print("FAQ search tool created!")

FAQ search tool created!


In [12]:
SYSTEM = """
You are the friendly assistant for Sweet Crumbs Home Bakery.

RULES:
1. Use get_price for prices.
2. Use is_open for opening timings.
3. Use search_faq for every policy question.
4. Answer policy questions only using retrieved information.
5. If information is not available, say:
   "I'll check with the owner."
6. Never make up information.
7. Before log_order, collect customer name,
   item, quantity and pickup date.
8. Ask for missing details before placing an order.
9. Keep answers under 60 words.
10. Reply in English or Telugu according to the customer.
"""

print("System prompt created!")

System prompt created!


In [13]:
memory = InMemorySaver()

agent = create_agent(
    model=llm,
    tools=[get_price, is_open, log_order, search_faq],
    system_prompt=SYSTEM,
    checkpointer=memory
)

print("LangGraph agent created successfully!")

LangGraph agent created successfully!


In [14]:
def chat(thread_id: str, text: str) -> str:

    config = {
        "configurable": {
            "thread_id": thread_id
        }
    }

    try:
        result = agent.invoke(
            {
                "messages": [
                    {"role": "user", "content": text}
                ]
            },
            config
        )

        return result["messages"][-1].content

    except Exception as e:
        print("Agent error:", e)
        return "Sorry, I'm a little busy. Please try again."

In [15]:
print("RAVI:")
print(chat("ravi", "Hi, I'm Ravi. Do you deliver? My order is about Rs 1300."))

print("\nSRAVANI:")
print(chat("sravani", "Hello! Do you make cupcakes?"))

print("\nRAVI AGAIN:")
print(chat("ravi", "Does the plum cake have nuts?"))

print("\nRAVI MEMORY TEST:")
print(chat("ravi", "What's my name?"))

RAVI:
Yes, we deliver within 5 km of the bakery. Delivery costs Rs 50, but it’s free for orders above Rs 1,000. To place your order, please let me know your name, the item(s), quantity, and pickup date.

SRAVANI:
Yes, we make cupcakes! Would you like to place an order?

RAVI AGAIN:
I’m not sure about that. I’ll check with the owner.

RAVI MEMORY TEST:
Your name is Ravi.


In [16]:
print(chat("sravani", "Do you sell biryani?"))

I’ll check with the owner.


In [17]:
%%writefile tests.csv
question,expected
Do you deliver?,5 km
Is the plum cake nut-free?,nuts
How much notice for a photo cake?,48
Can I pay by UPI?,UPI
Do you sell biryani?,check with the owner

Writing tests.csv


In [18]:
import pandas as pd

tests = pd.read_csv("tests.csv")

print(tests)
print("\nTotal tests:", len(tests))

                            question              expected
0                    Do you deliver?                  5 km
1         Is the plum cake nut-free?                  nuts
2  How much notice for a photo cake?                    48
3                  Can I pay by UPI?                   UPI
4               Do you sell biryani?  check with the owner

Total tests: 5


In [19]:
passed = 0

for index, row in tests.iterrows():

    response = chat(
        f"test_{index}",
        row["question"]
    )

    expected = row["expected"].lower()

    if expected in response.lower():
        passed += 1

    print("\nQuestion:", row["question"])
    print("Response:", response)
    print("Expected keyword:", expected)
    print("Result:", "PASS" if expected in response.lower() else "FAIL")

rate = (passed / len(tests)) * 100

print("\nPassed:", passed)
print("Total:", len(tests))
print("Pass rate:", rate, "%")


Question: Do you deliver?
Response: Yes, we deliver within 5 km of the bakery. Delivery costs Rs 50, free for orders above Rs 1000. Regular orders need a 24‑hour notice.
Expected keyword: 5 km
Result: FAIL

Question: Is the plum cake nut-free?
Response: Yes, the plum cake contains nuts and dried fruit. It is not nut‑free. (All other cakes, like cupcakes and chocolate cake, are nut‑free.)
Expected keyword: nuts
Result: PASS

Question: How much notice for a photo cake?
Response: Photo cakes need a 48‑hour notice before pickup.
Expected keyword: 48
Result: PASS

Question: Can I pay by UPI?
Response: Yes, we accept UPI payments. For orders above Rs 1000, a 50 % advance is required via UPI. (Refunds are not given after 12 hours before pickup.)
Expected keyword: upi
Result: PASS

Question: Do you sell biryani?
Response: I’ll check with the owner.
Expected keyword: check with the owner
Result: PASS

Passed: 4
Total: 5
Pass rate: 80.0 %
